# tools

LLMs are locked in a text-in/text-out vacuum with knowledge cutoffs. 
Tools are what give LLMs hands and eyes. In LangChain, a Tool wraps a specific action—like executing Python code, querying a SQL database, or searching the web. 
The key mechanism is that the LLM doesn't execute the code itself; it inspects the tool's description and outputs structured JSON requesting the environment to execute that tool on its behalf

models can request to call tools that preform tasks such as fetching data from a database,searching the web or running code

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]  =  os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("why it is difficult to get IT job ib 2026")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query**: The user is asking "why it is difficult to get IT job ib 2026". There\'s a typo ("ib" instead of "in"), but the meaning is clear: They want to know why getting an IT job is expected to be difficult in 2026.\n\n2.  **Identify Key Factors/Themes**: I need to analyze current and projected trends in the IT job market that would make job hunting difficult in 2026. Key areas to consider:\n   - AI automation & displacement\n   - Economic conditions (recessions, hiring freezes, budget constraints)\n   - Skill mismatch & evolving tech stack\n   - Market saturation & competition\n   - Remote work & geographic shifts\n   - Company restructuring & layoffs post-2022/2023 boom\n   - Regulatory/compliance changes\n   - Experience vs. entry-level gap\n\n3.  **Research/Verify Trends (Mental Knowledge up to 2024, projected to 2026)**:\n   - *AI Impact*: Generative AI is automating many entry/mid-level coding, tes

In [8]:
from langchain.tools import tool

@tool #@ is decorator -> which wraps the original function to add new functionallity without modefiying the org. function
def get_weather(location:str) -> str:
    """Get the weather at a location"""
    return f"its sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [9]:
response = model_with_tools.invoke("what's the weather like in mysuru")
print(response)
for tool_call in response.tool_calls:
    # view tool call made by the model
    print(f"tool:{tool_call['name']}")
    print(f"args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user wants to know the weather in "mysuru".\n2.  Identify the available tool: `get_weather` which takes a `location` parameter.\n3.  Extract the location: "mysuru" (Mysuru, also known as Mysore, Karnataka, India).\n4.  Call the tool: `get_weather(location="mysuru")`.\n5.  Wait for the tool response and formulate the answer.\n\nLet\'s call the tool. \nNote: I will use "Mysuru" as provided. I can also try "Mysuru, Karnataka" if needed, but usually just the city name works. I\'ll stick to "Mysuru".\nProceed. \nTool call: `get_weather(location="Mysuru")`\n', 'tool_calls': [{'id': 'qz3ff10fy', 'function': {'arguments': '{"location":"Mysuru"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 204, 'prompt_tokens': 276, 'total_tokens': 480, 'completion_time': 0.39072066, 'completion_tokens_details': {'reasoning_tokens': 174}, 'prompt_

In [ ]:
import requests
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """get the weather at a location"""
    # step 1 convert city name -> latitude/longitude
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name"     : location,
        "count"    : 1,
        "language" : "en",
        "format"   : "json"
    }

    geo_response = requests.get(geo_url,params=geo_params)
    geo_response.raise_for_status()

    geo_data = geo_response.json()

    if "results" not in geo_data:
        return f"could not find location:{location}"

    place = geo_data["results"][0]
    latitude = place["latitude"]
    longitude = place["longitude"]

# step 2 get weather data
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude" : longitude,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
        "timezone" : "auto"
    }

    weather_response = requests.get(
        weather_url,
        params=weather_params
    )

    weather_response.raise_for_status()
    weather_data = weather_response.json()
    current = weather_data["current"]

    return (
            f"Weather in {location}: "
            f"{current['temperature_2m']}°C, "
            f"humidity {current['relative_humidity_2m']}%, "
            f"wind speed {current['wind_speed_10m']} km/h."
        )




In [17]:
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke(
    "what is the weather like in mysuru?"
)

for tool_call in response.tool_calls:
    print(f"tool:{tool_call['name']}")
    print(f"args:{tool_call['args']}")

    result = get_weather.invoke(tool_call["args"])

    print("result:",result)

tool:get_weather
args:{'location': 'mysuru'}
result: Weather in mysuru: 27.3°C, humidity 67%, wind speed 22.6 km/h.


# tool execution loops

In [20]:
# step 1 model generate tool calls
messages = [{"role":"user","content":"what's the weather in mysuru"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step:2 exectue tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step:3 pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Mysuru is currently 26.7°C with 70% humidity and a wind speed of 21.1 km/h.
